<a href="https://colab.research.google.com/github/Nishikant090/Proactive-Handover-Trigger-Prediction-in-5G-Networks-Using-Time-Series-Forecasting-Models/blob/main/LSTM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import numpy as np
import tensorflow as tf

from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import GRU, Dense, Input
from tensorflow.keras.layers import MultiHeadAttention, LayerNormalization
from tensorflow.keras.layers import Dropout, GlobalAveragePooling1D

In [3]:
from google.colab import files
uploaded = files.upload()


Saving drive_test_measurements01.csv to drive_test_measurements01.csv
Saving drive_test_measurements02.csv to drive_test_measurements02.csv
Saving drive_test_measurements03.csv to drive_test_measurements03.csv


In [8]:
import pandas as pd

df1 = pd.read_csv("drive_test_measurements01.csv")
df2 = pd.read_csv("drive_test_measurements02.csv")
df3 = pd.read_csv("drive_test_measurements03.csv")

# Combine datasets
df = pd.concat([df1, df2, df3], ignore_index=True)

print("Combined shape:", df.shape)
df.head()


Combined shape: (8896, 11)


,Unnamed: 0,Lon,Lat,RSRP,RSRQ,CINR,PCI,Height (m),Distance (m),Azimuth,Elevation
0,0,-44.01509,-19.84822,-64.5,-10.5,-20.0,112,845.0,689.817854,136.575514,-3.623800
1,1,-44.01645,-19.84671,-64.5,-10.5,-20.0,112,852.1,907.680565,135.011892,-2.304621
2,2,-44.01684,-19.84581,-64.5,-10.5,-20.0,112,853.4,1007.502052,132.612617,-2.002201
3,3,-44.01687,-19.84574,-64.5,-10.5,-20.0,112,854.0,1015.292285,132.444430,-1.952953
4,4,-44.01725,-19.84487,-80.2,-10.4,-16.0,398,860.2,4505.743820,112.440028,-0.577324


In [9]:
print(df.columns)


Index(['Unnamed: 0', 'Lon', 'Lat', 'RSRP', 'RSRQ', 'CINR', 'PCI', 'Height (m)',
       'Distance (m)', 'Azimuth', 'Elevation'],
      dtype='object')


In [10]:
# Attempt common column names
possible_cols = ['RSRP','rssi','RSSI','Signal','signal','SINR']

signal_col = None
for c in possible_cols:
    if c in df.columns:
        signal_col = c
        break

# fallback: first numeric column
if signal_col is None:
    signal_col = df.select_dtypes(include='number').columns[0]

print("Using column:", signal_col)

data = df[signal_col].values.reshape(-1,1)


Using column: RSRP


In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)


In [12]:
import numpy as np

SEQ_LEN = 10
X=[]
y=[]

for i in range(len(data_scaled)-SEQ_LEN):
    X.append(data_scaled[i:i+SEQ_LEN])
    y.append(data_scaled[i+SEQ_LEN])

X=np.array(X)
y=np.array(y)

print("X shape:",X.shape)


X shape: (8886, 10, 1)


In [13]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense

lstm_model = Sequential([
    LSTM(64, input_shape=(X.shape[1], X.shape[2])),
    Dense(1)
])

lstm_model.compile(optimizer="adam", loss="mse")

lstm_model.fit(X,y,epochs=10,batch_size=32)


Epoch 1/10


/usr/local/lib/python3.12/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


278/278 ━━━━━━━━━━━━━━━━━━━━ 4s 9ms/step - loss: 0.0226
Epoch 2/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 4s 6ms/step - loss: 0.0038
Epoch 3/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0032
Epoch 4/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0029
Epoch 5/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0027
Epoch 6/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - loss: 0.0026
Epoch 7/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - loss: 0.0026
Epoch 8/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0026
Epoch 9/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0027
Epoch 10/10
278/278 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - loss: 0.0026


In [14]:
lstm_loss = lstm_model.evaluate(X,y)
print("LSTM Loss:", lstm_loss)

lstm_pred = lstm_model.predict(X)

# create labels
threshold = np.mean(y)
labels = (y < threshold).astype(int)

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X_train,X_test,y_train,y_test = train_test_split(
    lstm_pred,labels,test_size=0.2,random_state=42)

rf = RandomForestClassifier()
rf.fit(X_train,y_train)

pred = rf.predict(X_test)

print("LSTM Accuracy:", accuracy_score(y_test,pred))


278/278 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 0.0026
LSTM Loss: 0.0025061352644115686
278/278 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step


/usr/local/lib/python3.12/dist-packages/sklearn/base.py:1389: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return fit_method(estimator, *args, **kwargs)


LSTM Accuracy: 0.858830146231721
